# Demo of Word Embeddings for AI 5102

In this demo, we'll use **Gensim** to load pre-trained word vectors and explore what they can do.

We'll be using the **Google News word2vec vectors** — a set of 300-dimensional vectors trained on roughly 100 billion words from Google News articles. The word2vec algorithm learns vector representations of words by predicting surrounding words in a sentence. Words that appear in similar contexts end up with similar vectors.

*(To execute the code, just press the play button or hit Shift+Enter.)*

In [ ]:
!pip install -q gensim

In [ ]:
import gensim.downloader as api

print("Loading Google News word2vec vectors (~1.7 GB download on first run)...")
wv = api.load('word2vec-google-news-300')
print("Done!")

Now that the vectors are loaded, we can see how many words have vector representations. This is the size of our **vocabulary**. We can also check the **dimensionality** — the length of each vector.

In [ ]:
print(f"Vocabulary size: {len(wv):,} words")
print(f"Dimensionality: {wv.vector_size}")

In [ ]:
"cat" in wv

In [ ]:
"computer_programmer" in wv

In [ ]:
"schadenfreude" in wv

## What does a word vector look like?

We can print out what a vector looks like. It's just a bunch of real-valued numbers (positive or negative). The number of values is the dimensionality (300).

In [ ]:
wv["quiet"]

In [ ]:
wv.most_similar("quiet")

## Similarity

The cool thing about vectors is that they let us measure how similar two words are using **cosine similarity**. The result is a number between -1 and 1, with numbers closer to 1 meaning the words are more similar.

In [ ]:
wv.similarity("cats", "dogs")

Wait, isn't that comparing apples and oranges? No, but this is:

In [ ]:
wv.similarity("apples", "oranges")

In [ ]:
wv.similarity("professor", "cucumber")

In [ ]:
wv.similarity("raise", "fall")

## Most similar from a list

We can also query for the most similar word out of a given list of words.

In [ ]:
wv.most_similar_to_given("kittens", ["oranges", "strawberries", "tomatoes", "cats", "dogs", "trees"])

# Polysemous words

In early word embeddings, polysemous words were a problem. Polysemous words are words that have more than one meaning. For example, "bug" can mean:
* A creepy-crawly thing
* Something that makes you ill
* An error in your code
* A covert listening device
* (Verb) be annoying

In word2vec, all of these different meanings get averaged into one vector.

In [ ]:
wv.most_similar("bug", topn=20)

Polysemy is very common. For instance, "apple" could be the fruit or the company. Since the dominant sense of apple in our corpus is the fruit, its most similar vectors are fruits and not computers.

In [ ]:
wv.most_similar("apple", topn=20)

## Averaging vectors to shift meaning

We can average together vectors to steer toward a particular sense of a word.

In [ ]:
apple = wv["apple"]
computer = wv["computer"]
averaged = (apple + computer) / 2
wv.similar_by_vector(averaged, topn=20)

### What's happening under the hood

Here are the first few numbers in each vector, to make it easier to see the averaging:

In [ ]:
print("apple[:3]   =", apple[:3])
print("computer[:3] =", computer[:3])
print("averaged[:3] =", averaged[:3])

## Capitalization matters

Notice the difference between "apple" (the fruit) and "Apple" (the company):

In [ ]:
wv.most_similar("Apple", topn=20)

In [ ]:
wv.similar_by_vector((wv["college"] + wv["food"]) / 2, topn=20)

## Context-based embeddings

In subsequent models like ELMo and BERT, word embeddings were computed for sentences rather than individual words, allowing the surrounding words in the sentence to influence the vector for each word token. These context-based word embeddings allowed for unique word embeddings for each word instance.

# Solving word analogy problems

We can also test the analogy-solving capabilities of word vectors. For analogy problems like "***man*** is to ***king*** as ***woman*** is to **-----**" we use vector arithmetic. We take the vector for *king*, subtract the vector for *man*, and add the vector for *woman*:

+ *king*
- *man*
+ *woman*

The result is a vector. To figure out what word is closest to it, we find the most similar word vectors.

In [ ]:
wv.most_similar(positive=["king", "woman"], negative=["man"])

You can try other gender-based analogy problems like:

***man*** is to ***congressman*** as ***woman*** is to what?

***man*** is to ***father*** as ***woman*** is to what?

Try out other analogy problems on your own. Ones related to countries often work well.

In [ ]:
wv.most_similar(positive=["congressman", "woman"], negative=["man"])

In [ ]:
wv.most_similar(positive=["father", "woman"], negative=["man"])

## Country/city analogies

***London*** is to ***UK*** as ***Paris*** is to what?

In [ ]:
wv.most_similar(positive=["London", "France"], negative=["UK"])

## Adjective analogies

***small*** is to ***smaller*** as ***big*** is to what?

In [ ]:
wv.most_similar(positive=["smaller", "big"], negative=["small"])

In [ ]:
wv.most_similar(positive=["tiny", "big"], negative=["small"])

# Bias in word vectors

Negative societal biases appear in word vectors, since they are trained on data that contain those biases.

A classic example of bias in word analogy problems was demonstrated in [Bolukbasi et al. (2016)](https://arxiv.org/abs/1607.06520).

Outdated stereotypes of women are revealed in the word2vec embeddings when you use them to solve analogy problems like "***man*** is to ***computer programmer*** as ***woman*** is to **-----**":

In [ ]:
wv.most_similar(positive=["computer_programmer", "woman"], negative=["man"])